## EDA

In [15]:
import os
import typing
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

Немного пришечем данные: вынесем в отдельный столбец `mixed_class` флаг смешанного класса, `toxicity_class` строка-перечисление классов без префикса \_\_lable\_\_ через запятую, `message_text` - cам текст комментария.

In [16]:
def is_mixed_class(row: str) -> bool:
    sep_idx = row.index(" ")
    return len(row[:sep_idx].split(",")) > 1

def define_toxicity_class(row: str) -> str:
    sep_idx = row.index(" ")
    classes = row[:sep_idx].split(",")
    return ",".join([class_name[9:] for class_name in classes])

def extract_message_text(row: str) -> str:
    sep_idx = row.index(" ")
    return row[sep_idx + 1:]

In [17]:
lines = []
with open(os.path.expanduser("~/Desktop/dataset.txt"), "r") as file:
    lines = [line.rstrip() for line in file.readlines()]
data = pd.DataFrame(lines, columns=["raw_text"])
data["mixed_class"] = data["raw_text"].apply(is_mixed_class)
data["toxicity_class"] = data["raw_text"].apply(define_toxicity_class)
data["message_text"] = data["raw_text"].apply(extract_message_text)
data[["message_text", "toxicity_class", "mixed_class"]].describe()

,message_text,toxicity_class,mixed_class
count,248290,248290,248290
unique,248284,8,2
top,расстрелять нахуй,NORMAL,False
freq,2,203685,239957


In [18]:
data["mixed_class"].mean()

0.03356156107777196

Доля смешанных классов мала (всего 3%), поэтому в первом приближении можно пытаться решить задачу без них. Если потом появятся идеи как их можно использовать -- добавим обратно

In [19]:
clean_data = data[data["mixed_class"] == False][["message_text", "toxicity_class"]]
clean_data.drop_duplicates(subset=["message_text"], inplace=True)
clean_data["toxicity_class"].value_counts()

toxicity_class
NORMAL       203682
INSULT        28567
THREAT         5457
OBSCENITY      2245
Name: count, dtype: int64

Добавим примитивные численные признаки построенные по сообщению, чтобы оценить наличие выбросов по длине и составу символов.

In [20]:
clean_data["words"] = clean_data["message_text"].apply(lambda text: [word for word in text.split() if len(word) > 0])
clean_data["word_count"] = clean_data["words"].apply(lambda words: len(words))
clean_data["upper_case"] = clean_data["words"].apply(lambda words: any([word.isupper() for word in words]))
clean_data["symbol_count"] = clean_data["message_text"].apply(lambda text: sum([1 for symbol in text if not symbol.isspace()]))
clean_data["mean_word_len"] = clean_data["words"].apply(lambda words: np.array([len(word) for word in words]).mean())
clean_data["median_word_len"] = clean_data["words"].apply(lambda words: np.median(np.array([len(word) for word in words])))

In [21]:
clean_data

,message_text,toxicity_class,words,word_count,upper_case,symbol_count,mean_word_len,median_word_len
0,скотина! что сказать,INSULT,"[скотина!, что, сказать]",3,False,18,6.000000,7.0
1,я сегодня проезжала по рабочей и между домами ...,NORMAL,"[я, сегодня, проезжала, по, рабочей, и, между,...",29,False,152,5.241379,6.0
2,очередной лохотрон. зачем придумывать очередно...,NORMAL,"[очередной, лохотрон., зачем, придумывать, оче...",54,False,326,6.037037,6.0
3,"ретро дежавю ... сложно понять чужое сердце , ...",NORMAL,"[ретро, дежавю, ..., сложно, понять, чужое, се...",12,False,61,5.083333,6.0
4,а когда мы статус агрогородка получили?,NORMAL,"[а, когда, мы, статус, агрогородка, получили?]",6,False,34,5.666667,5.5
...,...,...,...,...,...,...,...,...
248285,правильно всё по пять (5)...,NORMAL,"[правильно, всё, по, пять, (5)...]",5,False,24,4.800000,4.0
248286,ёбанные нубы заходите на сервер мой ник _creep...,INSULT,"[ёбанные, нубы, заходите, на, сервер, мой, ник...",23,False,121,5.260870,4.0
248287,а у меня наверное рекорд в 1962 году в училище...,NORMAL,"[а, у, меня, наверное, рекорд, в, 1962, году, ...",25,False,164,6.560000,6.0
248288,спасибо всем большое),NORMAL,"[спасибо, всем, большое)]",3,False,19,6.333333,7.0
